In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pyspark")

current_path = os.getcwd()
project_root = Path(current_path).parent
os.chdir(project_root)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

cnefe_df = spark.read.parquet("data/integrated/integrated_cnefe_addresses.parquet")
cnes_df = spark.read.csv("data/downloads/cnes_estabelecimentos.csv", header=True, inferSchema=True, sep=";")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/28 07:26:42 WARN Utils: Your hostname, lucas-vital-Q570M-D3H, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp8s0)
26/07/28 07:26:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/lucas-vital/projects/brazilian_address_linkage/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/28 07:26:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/28

In [2]:
cnefe_df.printSchema()

root
 |-- COD_UNICO_ENDERECO: long (nullable = true)
 |-- COD_UF: integer (nullable = true)
 |-- COD_MUNICIPIO: long (nullable = true)
 |-- COD_DISTRITO: long (nullable = true)
 |-- COD_SUBDISTRITO: long (nullable = true)
 |-- COD_SETOR: string (nullable = true)
 |-- NUM_QUADRA: integer (nullable = true)
 |-- NUM_FACE: integer (nullable = true)
 |-- CEP: string (nullable = true)
 |-- DSC_LOCALIDADE: string (nullable = true)
 |-- NOM_TIPO_SEGLOGR: string (nullable = true)
 |-- NOM_TITULO_SEGLOGR: string (nullable = true)
 |-- NOM_SEGLOGR: string (nullable = true)
 |-- NUM_ENDERECO: integer (nullable = true)
 |-- DSC_MODIFICADOR: string (nullable = true)
 |-- NOM_COMP_ELEM1: string (nullable = true)
 |-- VAL_COMP_ELEM1: string (nullable = true)
 |-- NOM_COMP_ELEM2: string (nullable = true)
 |-- VAL_COMP_ELEM2: string (nullable = true)
 |-- NOM_COMP_ELEM3: string (nullable = true)
 |-- VAL_COMP_ELEM3: string (nullable = true)
 |-- NOM_COMP_ELEM4: string (nullable = true)
 |-- VAL_COMP_ELE

In [3]:
cnefe_df.select("DSC_ESTABELECIMENTO", "DSC_MODIFICADOR").distinct().toPandas().to_csv("data/outputs/cnefe_estabelecimentos.csv", index=False)

/home/lucas-vital/projects/brazilian_address_linkage/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
cnes_df.printSchema()

root
 |-- CO_CNES: integer (nullable = true)
 |-- CO_UNIDADE: string (nullable = true)
 |-- CO_UF: integer (nullable = true)
 |-- CO_IBGE: integer (nullable = true)
 |-- NU_CNPJ_MANTENEDORA: long (nullable = true)
 |-- NO_RAZAO_SOCIAL: string (nullable = true)
 |-- NO_FANTASIA: string (nullable = true)
 |-- CO_NATUREZA_ORGANIZACAO: integer (nullable = true)
 |-- DS_NATUREZA_ORGANIZACAO: string (nullable = true)
 |-- TP_GESTAO: string (nullable = true)
 |-- CO_NIVEL_HIERARQUIA: integer (nullable = true)
 |-- DS_NIVEL_HIERARQUIA: string (nullable = true)
 |-- CO_ESFERA_ADMINISTRATIVA: string (nullable = true)
 |-- DS_ESFERA_ADMINISTRATIVA: string (nullable = true)
 |-- CO_ATIVIDADE: integer (nullable = true)
 |-- TP_UNIDADE: integer (nullable = true)
 |-- CO_CEP: integer (nullable = true)
 |-- NO_LOGRADOURO: string (nullable = true)
 |-- NU_ENDERECO: string (nullable = true)
 |-- NO_BAIRRO: string (nullable = true)
 |-- NU_TELEFONE: string (nullable = true)
 |-- NU_LATITUDE: double (null

## Tentativas de pareamento

### Por latitude e longitude

In [5]:
from pyspark.sql import functions as F

cnes_df.filter(F.col("CO_CEP") == "29060120").show(11, truncate=False)

26/07/28 07:27:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------+-----+-------+-------------------+----------------------------------------------------------+------------------------------------------------+-----------------------+-----------------------+---------+-------------------+-------------------+------------------------+------------------------+------------+----------+--------+------------------------------------+-----------+----------------+--------------------+--------------+--------------+--------------------+----------------------------------------------+--------------+--------------------------------------+---------------+-------------------+--------------------+------------------+-------------------+----------------+---------------------+---------------+-------------------+
|CO_CNES|CO_UNIDADE   |CO_UF|CO_IBGE|NU_CNPJ_MANTENEDORA|NO_RAZAO_SOCIAL                                           |NO_FANTASIA                                     |CO_NATUREZA_ORGANIZACAO|DS_NATUREZA_ORGANIZACAO|TP_GESTAO|CO_NIVEL_HIERARQUIA|

In [6]:
joint_lat_long_df = cnes_df.join(cnefe_df, [cnes_df.NU_LATITUDE == cnefe_df.LATITUDE, cnes_df.NU_LONGITUDE == cnefe_df.LONGITUDE], "inner")
print("Número de registros encontrados no CNES: %d" % joint_lat_long_df.count())
print("Cobertura: %0.6f %%" % (joint_lat_long_df.count() / cnes_df.count() * 100))

Número de registros encontrados no CNES: 30


Cobertura: 0.004774 %


### Por nome do estabelecimento

In [7]:
joint_nome_df = cnes_df.join(cnefe_df, [cnes_df.NO_FANTASIA == cnefe_df.DSC_ESTABELECIMENTO ], "inner")
print("Número de registros encontrados no CNES: %d" % joint_nome_df.count())
print("Cobertura: %0.6f %%" % (joint_nome_df.count() / cnes_df.count() * 100))

Número de registros encontrados no CNES: 22213016


Cobertura: 3534.864210 %


### Por nome do estabelecimento e lougradouro

Aqui acredtio que consiga a cobertura a nível mais real

In [8]:
cnefe_df = cnefe_df.withColumn(
    "NO_LOGRADOURO",
    F.concat_ws(" ", F.col("NOM_TIPO_SEGLOGR"), F.col("NOM_TITULO_SEGLOGR"), F.col("NOM_SEGLOGR"))
)
joint_log_df = cnes_df.join(cnefe_df, [cnes_df.NO_FANTASIA == cnefe_df.DSC_ESTABELECIMENTO, cnes_df.NO_LOGRADOURO == cnefe_df.NO_LOGRADOURO ], "inner")
print("Número de registros encontrados no CNES: %d" % joint_log_df.count())
print("Cobertura: %0.6f %%" % (joint_log_df.count() / cnes_df.count() * 100))

Número de registros encontrados no CNES: 14507


Cobertura: 2.308569 %


### Por nome do estabelecimento e lougradouro e numero

Aqui acredito demonstrar que existam erros no registro do numero em pelo menos uma das bases

In [10]:
joint_log_num_df = cnes_df.join(
    cnefe_df,
    [
        cnes_df.NO_FANTASIA == cnefe_df.DSC_ESTABELECIMENTO,
        cnes_df.NO_LOGRADOURO == cnefe_df.NO_LOGRADOURO,
        F.col("NU_ENDERECO").cast("string") == F.col("NUM_ENDERECO").cast("string"),
    ],
    "inner"
)

print("Número de registros encontrados no CNES: %d" % joint_log_num_df.count())
print("Cobertura: %0.6f%%" % (joint_log_num_df.count() / cnes_df.count() * 100))

Número de registros encontrados no CNES: 6296


Cobertura: 1.001913%
